# VAE and conditional VAE representations

> **Notebook role:** production-style construction and interpretation of learned nuclear image embeddings.

## 1. Standardize nuclear crops

Crop size, channel count and intensity normalization are part of the model
contract. Object identifiers remain attached to every crop and latent row.

In [ ]:
from pathlib import Path
import numpy as np
import torch

crop_archive = np.load(Path("data/nuclear_crops.npz"), allow_pickle=False)
crops = crop_archive["images"]
object_ids = crop_archive["object_ids"].astype(str)
crop_tensor = torch.as_tensor(crops, dtype=torch.float32).reshape(-1, 1, 64, 64)
crop_tensor = crop_tensor / crop_tensor.amax(dim=(1, 2, 3), keepdim=True).clamp_min(1e-8)
crop_tensor.shape

## 2. Encode, sample and reconstruct

The VAE returns the reconstruction, sampled latent vector, latent mean and log
variance. The conditional model adds a classification objective while retaining
the same image representation path.

In [ ]:
from nuclear_vae_embeddings.models import CVAE, VAE

vae = VAE(nc=1, latent_variable_size=128, imsize=64)
cvae = CVAE(nc=1, latent_variable_size=128, imsize=64)
reconstruction, latent, mean, log_variance = vae(crop_tensor)
reconstruction.shape, latent.shape

## 3. Join learned and interpretable measurements

Latent dimensions complement morphology and texture. Keep object identifiers in
the embedding table so representations can be joined without changing row
identity.

In [ ]:
import pandas as pd

latent_table = pd.DataFrame(mean.detach().cpu().numpy()).add_prefix("latent_")
latent_table.insert(0, "object_id", object_ids)
nuclear_features = pd.read_csv(Path("outputs/features/nuclei.csv"))
combined = nuclear_features.merge(latent_table, on="object_id", validate="one_to_one")
combined.head()

## 4. Save the joined representation table

The joined table is the explicit hand-off to feature filtering, statistics and
figure generation.

In [ ]:
output_path = Path("outputs/representations.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
combined.to_csv(output_path, index=False)

## Representative result

A continuous embedding can be coloured by condition, timepoint, measured
feature or model prediction without changing the underlying coordinates.

![Latent and feature-space embedding](../assets/results/feature-embedding.png)